# 07 · End-to-end Inference & Submission

**Goals**
- Check which ground truth the competition scores against, and calibrate the post-processing to it
- Timed end-to-end inference on the 180 test images: JPEG → disk detection → preprocessing → network → instances
- Write `submission.csv` in the competition format and check every row before uploading

**Submission format** (competition Data page): one row per predicted filament,
`filament_id` = `<image id>_<k>`, `segmentation_rle` = COCO run-length encoding counts of a 2048 × 2048 mask
(`pycocotools.mask.encode`, counts only, no quotes).

**Inputs, all from previous notebooks:** the competition data, the preprocessed-images dataset (02), the
**output of notebook 04** (trained models) and the **output of notebook 05** (`postprocess.json`). On Kaggle, attach all
four (*Add Input*); turn on GPU and Internet. `COS711_SMOKE=1` checks the pipeline on a few images on CPU.

In [ ]:
# --- Setup: works both locally and on Kaggle ---
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    # Pull the shared src/ code from GitHub (requires Internet ON in notebook settings)
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

from src.data import get_data_dir
DATA_DIR = get_data_dir()
print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)

try:
    import segmentation_models_pytorch
except ImportError:
    %pip install -q segmentation-models-pytorch
try:
    import pycocotools
except ImportError:
    %pip install -q pycocotools

In [ ]:
import json, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

from src.data import ANNOTATION_FILE, load_annotations, list_test_images, load_image
from src.preprocess import find_disk, preprocess, load_preprocessed, disk_mask
from src.augment import NO_AUG
from src.dataset import FilamentDataset, model_input
from src.model import build_model
from src.train import make_loader, summarise
from src.postprocess import components, drop_small, upsample
from src.metrics import match_instances, pq_from_counts

SMOKE = os.environ.get("COS711_SMOKE") == "1"   # tiny CPU run: checks the pipeline, numbers meaningless
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

S = 1024              # working resolution (as notebook 04)
FULL = 2048           # image and submission resolution
LIMB_CUT = 0.985      # pixels beyond 0.985 R are never filament
VAL_FOLD = 0

if SMOKE:
    OUT_DIR = ROOT / "outputs" / "smoke"
    BATCH = 1
    WORKERS = 0
else:
    OUT_DIR = ROOT
    BATCH = 8
    WORKERS = min(4, os.cpu_count())

FIG_DIR = OUT_DIR / "figures" / "07"
ART_OUT = OUT_DIR / "artifacts"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ART_OUT.mkdir(parents=True, exist_ok=True)


def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)


def previous_output(rel):
    """A file written by an earlier notebook: an attached notebook output on Kaggle if there is one, else the repo copy."""
    candidates = []
    if IN_KAGGLE:
        candidates = sorted(Path("/kaggle/input").glob(f"**/{rel}"))
    candidates.append(OUT_DIR / rel)
    candidates.append(ROOT / rel)
    for path in candidates:
        if path.exists():
            print(rel, "←", path)
            return path
    message = f"{rel} not found: run the earlier notebooks first"
    if IN_KAGGLE:
        message += " and attach their outputs (Add Input → Notebook Output Files)"
    raise FileNotFoundError(message)


def preprocessed_dir(probe):
    """Folder that holds full_<S>/ with the preprocessed .npy files of notebook 02."""
    if IN_KAGGLE:
        candidates = []
        for f in sorted(Path("/kaggle/input").glob(f"**/{probe}")):
            candidates.append(f.parent)
    else:
        candidates = [ROOT / "cache" / f"full_{S}"]
    found = None
    for folder in candidates:
        if (folder / probe).exists():
            found = folder
            break
    if found is None:
        raise FileNotFoundError(f"preprocessed images ({probe}) not found: run notebook 02 / attach the dataset")
    print(f"cache/full_{S}", "←", found)
    # load_preprocessed reads <cache_dir>/full_<S>/<obs>.npy, so a folder with another name is linked under /tmp
    if found.name != f"full_{S}":
        link = Path("/tmp/cos711_cache") / f"full_{S}"
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            link.symlink_to(found, target_is_directory=True)
        found = link
    return found.parent


IMAGE_STATS_CSV = previous_output("artifacts/image_stats.csv")            # notebook 01
EXCLUDED_CSV = previous_output("artifacts/excluded_annotations.csv")      # notebook 02
FOLDS_CSV = previous_output("artifacts/folds.csv")                        # notebook 02
first_obs = pd.read_csv(FOLDS_CSV).obs.iloc[0]
CACHE_DIR = preprocessed_dir(f"{first_obs}.npy")                         # notebook 02
POSTPROCESS_JSON = previous_output("artifacts/postprocess.json")          # notebook 05

with open(POSTPROCESS_JSON) as fh:
    chosen = json.load(fh)
MODEL_NAME = chosen["model"]
USE_TTA = chosen["tta"]
SETTINGS = {"high": chosen["high"], "low": chosen["low"], "min_area": chosen["min_area"],
            "gap": chosen["gap"], "grow": chosen["grow"]}
CKPT = previous_output(f"checkpoints/{MODEL_NAME}.pt")                     # notebook 04
print()
print("model:", MODEL_NAME, "| TTA:", USE_TTA, "| post-processing:", SETTINGS)
if DEVICE.type == "cuda":
    print("device:", DEVICE, torch.cuda.get_device_name(), "| smoke:", SMOKE)
else:
    print("device:", DEVICE, "| smoke:", SMOKE)

## 1. Which ground truth does the competition score against?

The annotations are COCO polygons. Notebooks 01–05 rasterise them with OpenCV (`src.data.instance_mask`); the
organisers point to `pycocotools` (`annToMask`), which rasterises the same polygon slightly differently. Each
annotation's own `area` field tells which one the dataset was built with.

In [ ]:
with open(DATA_DIR / "train" / ANNOTATION_FILE) as fh:
    raw = json.load(fh)
segmentation = {}
area_field = {}
for a in raw["annotations"]:
    segmentation[a["id"]] = a["segmentation"]
    area_field[a["id"]] = a["area"]


def coco_mask(ann_id):
    """Boolean 2048 × 2048 mask of one annotation, rasterised by pycocotools (as the organisers do)."""
    rles = mask_utils.frPyObjects(segmentation[ann_id], FULL, FULL)
    return mask_utils.decode(mask_utils.merge(rles)) > 0


images, anns = load_annotations(DATA_DIR)
excluded = pd.read_csv(EXCLUDED_CSV)
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)

rows = []
for ann_id in anns.id.sample(200, random_state=0):
    coco = coco_mask(ann_id)
    poly = anns.loc[anns.id == ann_id, "polygon"].iloc[0]
    ours = cv2.fillPoly(np.zeros((FULL, FULL), np.uint8), [np.round(poly * 16).astype(np.int32)], 1, shift=4) > 0
    ys_c, xs_c = np.nonzero(coco)
    ys_o, xs_o = np.nonzero(ours)
    rows.append({"area field / pycocotools area": area_field[ann_id] / coco.sum(),
                 "area field / OpenCV area": area_field[ann_id] / ours.sum(),
                 "IoU pycocotools vs OpenCV": (coco & ours).sum() / (coco | ours).sum(),
                 "x shift (px)": xs_c.mean() - xs_o.mean(),
                 "y shift (px)": ys_c.mean() - ys_o.mean()})
pd.DataFrame(rows).median().to_frame("median over 200 filaments")

The `area` field matches the pycocotools masks exactly, so that is the ground truth the leaderboard uses. The
OpenCV masks the model was trained on are about 10% larger and shifted by about half a pixel down-right. For
filaments ~12 px wide that costs IoU, so below the validation predictions are re-scored against the pycocotools
ground truth, and two cheap corrections are tried without retraining: shifting the probability map by half a
pixel up-left, and a higher threshold (thinner masks).

## 2. Calibration on the validation fold

In [ ]:
folds = pd.read_csv(FOLDS_CSV)
geo_full = pd.read_csv(IMAGE_STATS_CSV).set_index("obs")[["cx", "cy", "radius"]]
paths = images.drop_duplicates("obs").set_index("obs").path
if SMOKE:
    folds = folds.groupby("fold").head(2).reset_index(drop=True)


def split(fold):
    train = folds.loc[folds.fold != fold, "obs"].tolist()
    val = folds.loc[folds.fold == fold, "obs"].tolist()
    return train, val


# Input normalisation exactly as in notebook 04 (same images, same code, so the same values)
train_obs, _ = split(VAL_FOLD)
probe = FilamentDataset(train_obs, images, anns, geo_full, size=S, cache_dir=CACHE_DIR)
step = max(1, len(probe.obs) // 100)
values = []
for o in probe.obs[::step]:
    cx, cy, r = geo_full.loc[o]
    x = load_preprocessed(paths[o], (cx, cy, r), S, cache_dir=CACHE_DIR)
    on_disk_pixels = x[disk_mask(x.shape, cx * S / FULL, cy * S / FULL, r * S / FULL, LIMB_CUT)]
    values.append(on_disk_pixels[::7])
values = np.concatenate(values)
med = float(np.median(values))
NORM = (med, float(1.4826 * np.median(np.abs(values - med))))
print("normalisation (centre, scale):", (round(NORM[0], 4), round(NORM[1], 4)))


def load_model(path):
    m = build_model(weights=None).to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    return m.eval()


def predict_batch(model, x, use_tta):
    """Filament probability for a batch x (B, 1, S, S); with TTA averaged over the 8 rotations / flips."""
    use_fp16 = DEVICE.type == "cuda"
    with torch.no_grad():
        if not use_tta:
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_fp16):
                return torch.sigmoid(model(x).float())
        total = 0
        for k in range(4):
            for flip in (False, True):
                view = torch.rot90(x, k, (2, 3))
                if flip:
                    view = view.flip(3)
                with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_fp16):
                    p = torch.sigmoid(model(view).float())
                if flip:
                    p = p.flip(3)
                total = total + torch.rot90(p, -k, (2, 3))
        return total / 8


def predict_dataset(model, dataset, use_tta):
    """{obs: probability at working resolution (float16)} for every image of a dataset."""
    loader = make_loader(dataset, BATCH, train=False, workers=WORKERS)
    out = {}
    for x, _, idx in tqdm(loader, desc="predict", leave=False):
        p = predict_batch(model, x.to(DEVICE), use_tta)[:, 0].cpu().numpy()
        ids = idx.tolist()
        for i in range(len(ids)):
            out[dataset.obs[ids[i]]] = p[i].astype(np.float16)
    return out


def coco_gt_fn(dataset):
    """Function giving the pycocotools instance maps (one per annotation set) of an observation."""
    def gt_fn(o):
        maps = []
        for image_id in dataset.set_ids[o]:
            m = np.zeros((FULL, FULL), np.int32)
            ann_ids = anns.loc[anns.image_id == image_id, "id"]
            k = 0
            for ann_id in ann_ids:   # later annotations win overlapping pixels, as in notebook 01
                k += 1
                m[coco_mask(ann_id)] = k
            maps.append(m)
        return maps
    return gt_fn


model = load_model(CKPT)
_, val_obs = split(VAL_FOLD)
va = FilamentDataset(val_obs, images, anns, geo_full, size=S, norm=NORM, aug=NO_AUG, cache_dir=CACHE_DIR)
probs = predict_dataset(model, va, USE_TTA)
print(len(probs), "validation images predicted")

In [ ]:
def disk_kernel(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def instances(p, on_disk, settings):
    """Instance map (0 = background) from a full-resolution probability map: notebook 05's post-processing."""
    high = settings["high"]
    low = settings["low"]
    if low is None:
        fg = (p > high) & on_disk
    else:
        fg = (p > low) & on_disk
    lab, _ = components(fg)
    if low is not None:
        # Hysteresis: keep a component of p > low only if it contains a pixel with p > high
        keep = np.zeros(lab.max() + 1, bool)
        keep[np.unique(lab[fg & (p > high)])] = True
        keep[0] = False
        fg = keep[lab]
    if settings["gap"] > 0:
        # Pieces closer than `gap` px form one instance; the pixels stay the predicted ones
        dilated = cv2.dilate(fg.astype(np.uint8), disk_kernel(settings["gap"])) > 0
        groups, _ = components(dilated)
        lab = np.where(fg, groups, 0)
    elif low is not None:
        lab, _ = components(fg)
    lab = drop_small(lab, np.bincount(lab.ravel()), settings["min_area"])
    if settings["grow"] > 0:
        grown = cv2.dilate(lab.astype(np.uint16), disk_kernel(settings["grow"])).astype(lab.dtype)
        lab = np.where((lab == 0) & on_disk, grown, lab)
    return lab


def full_res_probability(prob, shift):
    """Upsample a working-resolution probability map to 2048 px, optionally moved `shift` px up and left."""
    p = upsample(prob, FULL)
    if shift:
        M = np.float32([[1, 0, -shift], [0, 1, -shift]])
        p = cv2.warpAffine(p, M, (FULL, FULL), flags=cv2.INTER_LINEAR)
    return p


def score_variants(probs, variants, gt_fn, disks):
    """Pooled PQ / SQ / RQ for each variant {name: (shift, settings)}."""
    counts = {}
    for name in variants:
        counts[name] = np.zeros(4)
    for o in tqdm(probs, desc=f"{len(variants)} variants", leave=False):
        gts = gt_fn(o)
        cx, cy, r = disks[o]
        on_disk = disk_mask((FULL, FULL), cx, cy, r, LIMB_CUT)
        prob_by_shift = {}
        for name in variants:
            shift, settings = variants[name]
            if shift not in prob_by_shift:
                prob_by_shift[shift] = full_res_probability(probs[o], shift)
            pred = instances(prob_by_shift[shift], on_disk, settings)
            for gt in gts:
                m = match_instances(gt, pred)
                counts[name] += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
    rows = {}
    for name in variants:
        c = counts[name]
        rows[name] = pq_from_counts(int(c[0]), int(c[1]), int(c[2]), c[3])
    return pd.DataFrame(rows).T.sort_values("pq", ascending=False)


disks = {}
for o in va.obs:
    disks[o] = va.geo[o]


def our_gt_fn(o):
    return va.masks(o, FULL)


# The notebook 05 result, scored against our OpenCV ground truth and against the competition's
base = {"notebook 05 settings": (0.0, SETTINGS)}
print("PQ against our OpenCV ground truth:      ", round(score_variants(probs, base, our_gt_fn, disks).pq.iloc[0], 4))
print("PQ against the pycocotools ground truth: ", round(score_variants(probs, base, coco_gt_fn(va), disks).pq.iloc[0], 4))

In [ ]:
SHIFTS = (0.0, 0.5, 1.0)
HIGHS = (SETTINGS["high"], SETTINGS["high"] + 0.1, SETTINGS["high"] + 0.2)

variants = {}
for shift in SHIFTS:
    for h in HIGHS:
        settings = SETTINGS.copy()
        settings["high"] = round(min(h, 0.95), 2)
        variants[f"shift {shift} px, threshold {settings['high']}"] = (shift, settings)
calib = score_variants(probs, variants, coco_gt_fn(va), disks)
display(calib[["pq", "sq", "rq", "tp", "fp", "fn"]])

BEST_VARIANT = calib.index[0]
SHIFT, FINAL = variants[BEST_VARIANT]
print("chosen:", BEST_VARIANT, "| PQ", round(calib.pq.iloc[0], 4))

### Honest check on folds 1–4

The shift and threshold were chosen on fold 0. Notebook 04's cross-validation models (never used for any tuning) are
scored against the pycocotools ground truth with notebook 05's settings and with the calibrated ones.

In [ ]:
cv_rows = []
for f in sorted(folds.fold.unique()):
    if f == VAL_FOLD:
        continue
    try:
        ckpt = previous_output(f"checkpoints/{MODEL_NAME}_fold{f}.pt")
    except FileNotFoundError:
        print(f"fold {f}: no checkpoint, skipped")
        continue
    m = load_model(ckpt)
    _, obs_f = split(f)
    va_f = FilamentDataset(obs_f, images, anns, geo_full, size=S, norm=NORM, aug=NO_AUG, cache_dir=CACHE_DIR)
    disks_f = {}
    for o in va_f.obs:
        disks_f[o] = va_f.geo[o]
    pr = predict_dataset(m, va_f, USE_TTA)
    two = {"notebook 05": (0.0, SETTINGS), "calibrated": (SHIFT, FINAL)}
    res = score_variants(pr, two, coco_gt_fn(va_f), disks_f)
    before = res.loc["notebook 05", "pq"]
    after = res.loc["calibrated", "pq"]
    cv_rows.append({"fold": f, "notebook 05": before, "calibrated": after, "gain": after - before})
    del m
    torch.cuda.empty_cache()

if cv_rows:
    cv = pd.DataFrame(cv_rows).set_index("fold")
    print(f"folds {list(cv.index)} (pycocotools ground truth): notebook 05 {round(cv['notebook 05'].mean(), 3)} ± "
          f"{round(cv['notebook 05'].std(), 3)} → calibrated {round(cv.calibrated.mean(), 3)} ± {round(cv.calibrated.std(), 3)}"
          f"  (gain {round(cv.gain.mean(), 3)} ± {round(cv.gain.std(), 3)})")
    display(cv)
    if cv.gain.mean() <= 0:
        # The correction does not generalise: keep notebook 05's settings
        SHIFT, FINAL = 0.0, SETTINGS
        print("calibration does not help on unseen folds → using notebook 05's settings")
else:
    cv = None
print("final: shift", SHIFT, "px | settings", FINAL, "| TTA", USE_TTA)

### The RLE encoding round trip

Every validation prediction is encoded exactly as for the submission, decoded again and re-scored: the PQ must not
change.

In [ ]:
def encode(mask):
    """COCO RLE counts string of a boolean 2048 × 2048 mask."""
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def decode(counts):
    return mask_utils.decode({"size": [FULL, FULL], "counts": counts.encode("ascii")}) > 0


gt_fn = coco_gt_fn(va)
direct = np.zeros(4)
round_trip = np.zeros(4)
for o in tqdm(list(probs)[:40], desc="RLE round trip", leave=False):
    cx, cy, r = disks[o]
    pred = instances(full_res_probability(probs[o], SHIFT), disk_mask((FULL, FULL), cx, cy, r, LIMB_CUT), FINAL)
    rebuilt = np.zeros_like(pred)
    for k in range(1, pred.max() + 1):
        rebuilt[decode(encode(pred == k))] = k
    for gt in gt_fn(o):
        m = match_instances(gt, pred)
        direct += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
        m = match_instances(gt, rebuilt)
        round_trip += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
pq_direct = pq_from_counts(int(direct[0]), int(direct[1]), int(direct[2]), direct[3])["pq"]
pq_rle = pq_from_counts(int(round_trip[0]), int(round_trip[1]), int(round_trip[2]), round_trip[3])["pq"]
print("PQ on 40 images: direct", round(pq_direct, 6), "| after encode → decode", round(pq_rle, 6))
assert abs(pq_direct - pq_rle) < 1e-9, "the RLE round trip changed the masks"

## 3. End-to-end inference on the test set

From the JPEG on disk to the instance map, nothing cached: load, disk detection, preprocessing, network (with TTA if
notebook 05 chose it), post-processing, RLE encoding. Each step is timed.

In [ ]:
test = list_test_images(DATA_DIR)
if SMOKE:
    test = test.head(3)


def end_to_end(path):
    """JPEG → instance map at 2048 px, with the time of each step (s)."""
    t = {}

    t0 = time.perf_counter()
    img = load_image(path)
    t["load"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    disk = find_disk(img)
    x = cv2.resize(preprocess(img, "full", disk), (S, S), interpolation=cv2.INTER_AREA)
    disk_small = (disk[0] * S / FULL, disk[1] * S / FULL, disk[2] * S / FULL)
    x = model_input(x, disk_small, NORM, 1.0)
    t["preprocess"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    batch = torch.from_numpy(x).reshape(1, 1, S, S).to(DEVICE)
    prob = predict_batch(model, batch, USE_TTA)[0, 0].cpu().numpy()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t["network"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    on_disk = disk_mask((FULL, FULL), disk[0], disk[1], disk[2], LIMB_CUT)
    lab = instances(full_res_probability(prob, SHIFT), on_disk, FINAL)
    t["postprocess"] = time.perf_counter() - t0
    return lab, disk, t


end_to_end(test.path.iloc[0])   # warm-up (cuDNN autotuning, lazy init)

rows = []
timings = []
predictions = {}
for i in tqdm(range(len(test)), desc="test images"):
    obs = test.obs.iloc[i]
    lab, disk, t = end_to_end(test.path.iloc[i])
    t0 = time.perf_counter()
    for k in range(1, lab.max() + 1):
        rows.append({"filament_id": f"{obs}_{k}", "segmentation_rle": encode(lab == k)})
    t["encode"] = time.perf_counter() - t0
    timings.append(t)
    predictions[obs] = (lab, disk)

timing = pd.DataFrame(timings)
timing["total"] = timing.sum(axis=1)
print(len(test), "test images,", len(rows), "predicted filaments,",
      round(timing.total.sum(), 1), "s end to end (", round(1000 * timing.total.mean()), "ms per image )")
(1000 * timing.mean()).round(1).to_frame("ms / image")

## 4. Write and check `submission.csv`

In [ ]:
submission = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
if IN_KAGGLE:
    SUBMISSION = Path("/kaggle/working/submission.csv")   # where Kaggle looks for a notebook's submission
else:
    SUBMISSION = OUT_DIR / "submission.csv"
submission.to_csv(SUBMISSION, index=False)
print("written:", SUBMISSION, "|", len(submission), "rows")

# Read the file back and check every row the way the scorer will read it
check = pd.read_csv(SUBMISSION)
assert list(check.columns) == ["filament_id", "segmentation_rle"]
assert check.filament_id.is_unique, "duplicate filament_id"
image_ids = check.filament_id.str.rsplit("_", n=1).str[0]
assert set(image_ids) <= set(test.obs), "filament_id with an unknown image id"
areas = []
for counts in tqdm(check.segmentation_rle, desc="decode check", leave=False):
    mask = decode(counts)
    assert mask.shape == (FULL, FULL)
    areas.append(int(mask.sum()))
assert min(areas) > 0, "empty mask in the submission"

per_image = image_ids.value_counts()
no_prediction = sorted(set(test.obs) - set(image_ids))
print("all rows decode to non-empty 2048 × 2048 masks")
print("filaments per image: median", int(per_image.median()), "| max", int(per_image.max()),
      "| images with no predicted filament:", len(no_prediction))
check.head()

In [ ]:
# A sample of test predictions (no labels exist for the test set)
sample = test.sample(min(6, len(test)), random_state=0).obs.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(len(axes.flat)):
    ax = axes.flat[i]
    ax.set_axis_off()
    if i >= len(sample):
        continue
    obs = sample[i]
    lab, disk = predictions[obs]
    img = load_image(test.loc[test.obs == obs, "path"].iloc[0])
    x = cv2.resize(preprocess(img, "full", disk), (S, S), interpolation=cv2.INTER_AREA)
    small = cv2.resize(lab.astype(np.float32), (S, S), interpolation=cv2.INTER_NEAREST)
    ax.imshow(x, cmap="gray", vmin=0.8, vmax=1.2)
    ax.contour((small > 0).astype(float), levels=[0.5], colors="red", linewidths=0.6)
    ax.set_title(f"{obs}: {int(lab.max())} filaments", fontsize=9)
fig.suptitle("Test images with predicted filaments (red)", y=0.99)
plt.tight_layout()
savefig("test_examples")
plt.show()

## Summary

In [ ]:
summary = {
    "model": MODEL_NAME,
    "post-processing": f"threshold {FINAL['high']}, min_area {FINAL['min_area']}, gap {FINAL['gap']}, shift {SHIFT} px",
    "TTA": USE_TTA,
    "validation PQ, fold 0 (pycocotools ground truth)": round(calib.loc[BEST_VARIANT, "pq"], 4),
    "test images": len(test),
    "predicted filaments": len(submission),
    "end-to-end time, all test images (s)": round(timing.total.sum(), 1),
    "end-to-end time per image (ms)": round(1000 * timing.total.mean(), 1),
    "device": DEVICE.type,
}
if cv is not None:
    summary["PQ folds 1–4, calibrated (pycocotools ground truth)"] = f"{round(cv.calibrated.mean(), 3)} ± {round(cv.calibrated.std(), 3)}"
display(pd.Series(summary).to_frame("value"))

used = {"model": MODEL_NAME, "tta": bool(USE_TTA), "shift": SHIFT}
used.update(FINAL)
with open(ART_OUT / "inference.json", "w") as fh:
    json.dump(used, fh, indent=1)